# Market Data Pipeline - Equity Trading Cost Analysis

Fetch, clean, and prepare intraday market data for transaction cost analysis and execution simulation.

## Setup

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings('ignore')

## Trading Universe

Using 10 liquid US equities across different sectors.

In [2]:
TICKERS = [
    'SPY', 'AAPL', 'MSFT', 'JPM', 'BAC',
    'XOM', 'JNJ', 'GOOGL', 'TSLA', 'WMT'
]

print(f"Trading Universe: {len(TICKERS)} stocks")
print(f"Tickers: {', '.join(TICKERS)}")

Trading Universe: 10 stocks
Tickers: SPY, AAPL, MSFT, JPM, BAC, XOM, JNJ, GOOGL, TSLA, WMT


## Fetch Intraday Market Data

Pulling 5-minute bars for the last 60 days to support realistic execution window analysis.

In [ ]:
def fetch_intraday_data(ticker, period='60d', interval='5m'):
    """Fetch intraday data from Yahoo Finance."""
    print(f"Fetching {ticker}...")
    
    try:
        df = yf.download(ticker, period=period, interval=interval, progress=False)
        
        if df.empty:
            print(f"  No data returned for {ticker}")
            return None
        
        # Clean column names
        df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]
        df['Ticker'] = ticker
        df = df.reset_index()
        
        print(f"  {ticker}: {len(df)} bars")
        return df
        
    except Exception as e:
        print(f"  Error fetching {ticker}: {str(e)}")
        return None


In [ ]:
# Fetch data for all tickers
all_data = []
for ticker in TICKERS:
    df = fetch_intraday_data(ticker)
    if df is not None:
        all_data.append(df)

market_data = pd.concat(all_data, ignore_index=True)

print(f"\nFetched {len(TICKERS)} tickers")
print(f"Total bars: {len(market_data):,}")
print(f"Date range: {market_data['Datetime'].min()} to {market_data['Datetime'].max()}")


## Convert to Eastern Time and Filter Market Hours

In [ ]:
# Convert to Eastern Time and filter regular trading hours (9:30 - 16:00 ET)
market_data['Datetime'] = pd.to_datetime(market_data['Datetime'], utc=True)
market_data['Datetime'] = market_data['Datetime'].dt.tz_convert('America/New_York')
market_data['Datetime'] = market_data['Datetime'].dt.tz_localize(None)

# Extract hour and minute
market_data['Hour'] = market_data['Datetime'].dt.hour
market_data['Minute'] = market_data['Datetime'].dt.minute

# Filter market hours
market_data = market_data[
    ((market_data['Hour'] == 9) & (market_data['Minute'] >= 30)) |
    ((market_data['Hour'] >= 10) & (market_data['Hour'] < 16)) |
    ((market_data['Hour'] == 16) & (market_data['Minute'] == 0))
].copy()

print(f"Market hours filtered: {len(market_data):,} bars remaining")

In [7]:
# Quick verification
print("\nVerification - Hour distribution:")
print(market_data['Hour'].value_counts().sort_index())
print("\nSample timestamps:")
print(market_data[['Datetime', 'Ticker', 'Hour', 'Minute']].head(10))


Verification - Hour distribution:
Hour
9     3600
10    7200
11    7197
12    7178
13    6960
14    6930
15    6840
Name: count, dtype: int64

Sample timestamps:
             Datetime Ticker  Hour  Minute
0 2025-11-10 09:30:00    SPY     9      30
1 2025-11-10 09:35:00    SPY     9      35
2 2025-11-10 09:40:00    SPY     9      40
3 2025-11-10 09:45:00    SPY     9      45
4 2025-11-10 09:50:00    SPY     9      50
5 2025-11-10 09:55:00    SPY     9      55
6 2025-11-10 10:00:00    SPY    10       0
7 2025-11-10 10:05:00    SPY    10       5
8 2025-11-10 10:10:00    SPY    10      10
9 2025-11-10 10:15:00    SPY    10      15


## Calculate VWAP and TWAP

In [ ]:
def calculate_vwap_twap(df):
    """Calculate daily VWAP and TWAP."""
    df = df.copy()
    df['Date'] = df['Datetime'].dt.date
    df['TypicalPrice'] = (df['High'] + df['Low'] + df['Close']) / 3
    df['PV'] = df['TypicalPrice'] * df['Volume']
    
    daily = df.groupby(['Ticker', 'Date']).apply(
        lambda x: pd.Series({
            'VWAP': (x['PV'].sum() / x['Volume'].sum()) if x['Volume'].sum() > 0 else x['TypicalPrice'].mean(),
            'TWAP': x['TypicalPrice'].mean()
        })
    ).reset_index()
    
    df = df.merge(daily, on=['Ticker', 'Date'], how='left')
    return df

market_data = calculate_vwap_twap(market_data)
print("VWAP and TWAP calculated")

## Add Trading Features

In [ ]:
def add_trading_features(df):
    df = df.copy()
    df['DayOfWeek'] = df['Datetime'].dt.dayofweek
    df['DayName'] = df['Datetime'].dt.day_name()
    
    def get_session(hour, minute):
        if (hour == 9 and minute >= 30) or hour == 10:
            return 'Market Open'
        elif hour >= 15:
            return 'Market Close'
        else:
            return 'Mid-Day'
    
    df['Session'] = df.apply(lambda x: get_session(x['Hour'], x['Minute']), axis=1)
    df['PriceRangePct'] = ((df['High'] - df['Low']) / df['Close']) * 100
    
    # Average daily volatility per ticker/date
    vol = df.groupby(['Ticker', 'Date'])['PriceRangePct'].mean().reset_index()
    vol.columns = ['Ticker', 'Date', 'AvgVolatility']
    df = df.merge(vol, on=['Ticker', 'Date'], how='left')
    
    return df

market_data = add_trading_features(market_data)
print("Trading features added")

## Calculate Average Daily Volume & Save Data

In [ ]:
# Average Daily Volume
avg_daily_volume = market_data.groupby('Ticker')['Volume'].sum() / market_data.groupby('Ticker')['Date'].nunique()
avg_daily_volume = avg_daily_volume.reset_index(name='AvgDailyVolume')

market_data = market_data.merge(avg_daily_volume, on='Ticker', how='left')

# Save processed data
market_data.to_csv('market_data_processed.csv', index=False)
print(f"Saved {len(market_data):,} records to market_data_processed.csv")

## Final Data Quality Check

In [9]:
print("DATA QUALITY SUMMARY")
print("=" * 50)
print(f"Total records : {len(market_data):,}")
print(f"Date range    : {market_data['Datetime'].min().date()} to {market_data['Datetime'].max().date()}")
print(f"Tickers       : {market_data['Ticker'].nunique()}")
print(f"Missing values: {market_data.isnull().sum().sum()}")

DATA QUALITY SUMMARY
Total records : 46,797
Date range    : 2026-01-02 to 2026-03-30
Tickers       : 10
Missing values: 0
